# Question -1 
- Load the dataset and replace all “?” values with appropriate missing values.
- Convert columns into correct data types (numeric, categorical, datetime).
- Handle missing values using appropriate techniques and explain your approach.
- Identify and treat outliers in:
    - appliance_usage_hours
    - ac_usage_hours
    - energy_consumption_kwh
- Create at least 3 new features such as:
    - total_usage_hours
    - temperature_effect
    - net_energy_usage
- Explain how each created feature helps in understanding energy consumption behavior.

In [1]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('smart_home_energy_dataset.csv')

# Replace '?' with NaN
df.replace('?', np.nan, inplace=True)

# Check missing values
print(df.isnull().sum())

date                      22
household_id              16
num_occupants             16
avg_temperature_c         19
appliance_usage_hours     14
ac_usage_hours            18
heater_usage_hours        15
solar_generation_kwh      23
humidity_pct              17
weekday                   15
electricity_price         24
peak_hour_usage           10
energy_consumption_kwh    21
dtype: int64


In [17]:
# Convert numeric columns
numeric_cols = [
    'appliance_usage_hours',
    'ac_usage_hours',
    'heater_usage_hours',
    'energy_consumption_kwh',
    'avg_temperature_c', 
    'humidity_pct',
    'solar_generation_kwh',
    'electricity_price',
    'num_occupants'
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Convert categorical columns
categorical_cols = ['weekday', 'peak_hour_usage']

for col in categorical_cols:
    df[col] = df[col].astype('category')

# Convert datetime column
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Verify
print(df.dtypes)

date                      datetime64[ns]
household_id                      object
num_occupants                    float64
avg_temperature_c                float64
appliance_usage_hours            float64
ac_usage_hours                   float64
heater_usage_hours               float64
solar_generation_kwh             float64
humidity_pct                     float64
weekday                         category
electricity_price                float64
peak_hour_usage                 category
energy_consumption_kwh           float64
dtype: object


In [25]:
# Numeric columns → fill with median
for col in numeric_cols:
    df[col].fillna(df[col].median(), inplace=True)

# Categorical columns → fill with mode
for col in categorical_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

# Datetime → drop rows if missing (rare case)
df.dropna(subset=['date'], inplace=True)

df.dropna(inplace=True)

C:\Users\karan\AppData\Local\Temp\ipykernel_26852\2768127400.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
C:\Users\karan\AppData\Local\Temp\ipykernel_26852\2768127400.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For examp

In [19]:
import numpy as np

def treat_outliers(col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    # Cap the outliers
    df[col] = np.where(df[col] < lower, lower,
                       np.where(df[col] > upper, upper, df[col]))

# Apply to required columns
outlier_cols = ['appliance_usage_hours', 'ac_usage_hours', 'energy_consumption_kwh']

for col in outlier_cols:
    treat_outliers(col)

In [20]:
df.dtypes

date                      datetime64[ns]
household_id                      object
num_occupants                    float64
avg_temperature_c                float64
appliance_usage_hours            float64
ac_usage_hours                   float64
heater_usage_hours               float64
solar_generation_kwh             float64
humidity_pct                     float64
weekday                         category
electricity_price                float64
peak_hour_usage                 category
energy_consumption_kwh           float64
dtype: object

In [21]:
df['total_usage_hours'] = (
    df['appliance_usage_hours'] +
    df['ac_usage_hours'] +
    df['heater_usage_hours']
)

In [22]:
df['temperature_effect'] = df['avg_temperature_c'] * df['ac_usage_hours']

In [23]:
df['net_energy_usage'] = (
    df['energy_consumption_kwh'] -
    df['solar_generation_kwh']
)

In [24]:
print(df.head())
print(df.describe())

        date household_id  num_occupants  avg_temperature_c  \
0 2024-01-01           H4            3.0          34.168340   
1 2024-01-02           H5            4.0          24.648470   
2 2024-01-03           H1            5.0          17.425797   
3 2024-01-04           H2            5.0          10.323154   
4 2024-01-05           H4            3.0          17.407615   

   appliance_usage_hours  ac_usage_hours  heater_usage_hours  \
0               3.067518        8.883962            3.149001   
1               1.036225        2.641536            3.902248   
2              11.727061        0.641933            0.448397   
3               7.971690        1.421679            1.856783   
4              12.564465        2.758374            6.817476   

   solar_generation_kwh  humidity_pct  weekday  electricity_price  \
0              2.267055     34.059406  Weekend           7.671870   
1              7.169379     74.918097  Weekend           7.034067   
2              5.422963     8